# Step 08D — Frozen model after 33 expert-approved corrections

**No retraining.** Paired, validator-aligned reference standard sensitivity evaluation for source-defined GT, missing lesions corrected in 11 images, and combined corrections of 33 images (11 + 22). Original files and checkpoints are never modified.

**Clinical caveat:** user reports both dentists approved the exported boxes. Preserve their written approval. Deleting all boxes in seven images requires particular clinical sign-off; the notebook logs those images and never silently changes original GT. These corrections are limited to 33 of 96 hold-out images; do not claim a completely reannotated reference standard.

**Data use:** radiographs are not included in the output input bundle. Use original source licenses to determine whether preview images may be shared.

In [ ]:
%pip -q install "ultralytics==8.4.139" numpy pandas matplotlib pyyaml pillow scipy


## 1. Verify frozen model and four exact CVAT exports; upload one input ZIP
Use the supplied `Step_08D_Final_33_Approved_CVAT_Inputs.zip` file. Colab renaming with `(2)` etc is supported automatically.

In [ ]:
from google.colab import drive,files
from pathlib import Path
from datetime import datetime,timezone
import hashlib, zipfile, json, io, os, shutil, platform, sys, yaml
from collections import Counter
import numpy as np, pandas as pd, torch, ultralytics
from ultralytics import YOLO
from ultralytics.utils.metrics import ap_per_class
from PIL import Image,ImageDraw

drive.mount('/content/drive')
assert ultralytics.__version__ == '8.4.139',ultralytics.__version__
assert torch.cuda.is_available(), 'Select T4 GPU in Colab runtime settings.'
ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
WEIGHTS=ROOT/'final_model/YOLO11s_Final/final_60epochs.pt'
EXPECTED_MODEL='19ce0e2ae9731a9acb5baeaa4cb514782aa182e7d916d8a101298dd59cdb077e'
ORIGINAL={'internal':ROOT/'processed_data/yolo_caries_baseline',
          'external':ROOT/'external_validation/MOPG7_66'}
EXPECTED_COUNTS={'internal':(30,108),'external':(66,55)}
MISSING_NAMES={'internal':{'cate1_009.png','cate1_010.png','cate2_010.png'},
               'external':{'00001247.jpg','00001373.jpg','00001381.jpg','0000538.jpg','0000623.jpg','ibn159.jpg','ibn285.jpg','ibn59.jpg'}}
EXPECTED_ADDED={'internal':(3,7),'external':(8,15)}
ZIP_NAMES={'internal':('Internal_3_CVAT_Annotations.zip','Internal_20_CVAT_Final_Annotations.zip'),
           'external':('External_8_CVAT_Annotations.zip','External_2_CVAT_Final_Annotations.zip')}
EXPECTED_ZIP_HASH={'Internal_3_CVAT_Annotations.zip':'27e8238dbaf2cb48fa6d75eb8eeeca9567ddd6fd0e77db83ed101d43c1bb86e9',
                   'External_8_CVAT_Annotations.zip':'634478697e7f8e6842afe00db7ea6f13165e9149b811e2df3f30bc666ebfb347',
                   'Internal_20_CVAT_Final_Annotations.zip':'d5ff1acee421a8e219170d492bfe831a5ee602a4c3c6e8ff757a2114490a540e',
                   'External_2_CVAT_Final_Annotations.zip':'d9d767deede20d7ad32ee80114c29d82f6e8be2cdb08eb55c53db629fb30b57f'}
REQUIRED_MEMBERS=set(EXPECTED_ZIP_HASH)|{'Step01B_Original_96_SHA256_Manifest.csv','Step08C_Expected_Source_Box_Counts.csv'}
def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
 return h.hexdigest()
assert sha256(WEIGHTS)==EXPECTED_MODEL,'STOP: Frozen model SHA256 mismatch.'
print('Model hash verified:',EXPECTED_MODEL,'GPU:',torch.cuda.get_device_name(0))

def approved_bundle(p):
 try:
  with zipfile.ZipFile(p) as z:
   return REQUIRED_MEMBERS<=set(z.namelist()) and z.testzip() is None
 except Exception: return False
candidates=sorted(Path('/content').glob('Step_08D_Final_33_Approved_CVAT_Inputs*.zip'),key=lambda p:p.stat().st_mtime,reverse=True)
valid=[p for p in candidates if approved_bundle(p)]
if valid:
 SOURCE_ARCHIVE=valid[0];print('Reused uploaded archive',SOURCE_ARCHIVE)
else:
 print('Upload Step_08D_Final_33_Approved_CVAT_Inputs.zip, even if renamed by Colab:')
 uploaded=files.upload()
 valid=[(k,v) for k,v in uploaded.items() if k.startswith('Step_08D_Final_33_Approved_CVAT_Inputs') and k.lower().endswith('.zip')]
 assert len(valid)==1,('Wrong input selected',list(uploaded))
 SOURCE_ARCHIVE=Path('/content/Step_08D_Final_33_Approved_CVAT_Inputs.zip')
 SOURCE_ARCHIVE.write_bytes(valid[0][1])
 assert approved_bundle(SOURCE_ARCHIVE),'STOP: archive has missing files or corrupt ZIP.'
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'Step_08D_Final_Expert_GT_Analysis_{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
INPUT_DIR=Path('/content/step08d_inputs')
INPUT_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as z:
 for name in REQUIRED_MEMBERS:
  (INPUT_DIR/name).write_bytes(z.read(name))
for name,h in EXPECTED_ZIP_HASH.items():
 assert sha256(INPUT_DIR/name)==h,('STOP: expert-approved CVAT export not identical to audited snapshot',name)
ORIGINAL_MANIFEST=pd.read_csv(INPUT_DIR/'Step01B_Original_96_SHA256_Manifest.csv')
assert len(ORIGINAL_MANIFEST)==96 and len(ORIGINAL_MANIFEST[['cohort','image']].drop_duplicates())==96
EXPECTED_SOURCE=pd.read_csv(INPUT_DIR/'Step08C_Expected_Source_Box_Counts.csv')
assert len(EXPECTED_SOURCE)==22
print('PASS: four exact CVAT archive hashes, 96 original manifest rows, 22 flagged cases.')
print('Output folder:',OUT)


## 2. Verify 33 approved images, exact input hashes, geometry, and unchanged 96 source images/labels
All fully negative corrected images are represented by **explicit zero-byte YOLO TXT files** in the temporary evaluation set. The CVAT train.txt manifest must list all 20 images, even when only 13 TXT label files are included.

In [ ]:
# Read source-flagged correction exports. Missing label file = deliberately empty image,
# permitted ONLY if that image occurs in the CVAT train.txt manifest.
def import_cvat_zip(path):
 with zipfile.ZipFile(path) as z:
  assert z.testzip() is None
  names=z.namelist()
  assert 'train.txt' in names and 'data.yaml' in names
  import_names=[Path(line.strip()).name for line in z.read('train.txt').decode().splitlines() if line.strip()]
  assert len(set(import_names))==len(import_names),'Duplicate filenames in CVAT train.txt.'
  labels={Path(n).stem:z.read(n) for n in names if n.startswith('labels/train/') and n.endswith('.txt')}
  assert len(labels)==len([n for n in names if n.startswith('labels/train/') and n.endswith('.txt')])
  assert set(labels)<={Path(n).stem for n in import_names},'Orphan label in CVAT export.'
  def check_label(image,blob):
   rows=[x for x in blob.decode('utf-8').splitlines() if x.strip()]
   for row in rows:
    parts=row.split();assert len(parts)==5,(image,row)
    a=np.asarray([float(x) for x in parts]); assert np.isfinite(a).all() and a[0]==0,(image,row)
    _,x,y,w,h=a
    assert w>0 and h>0 and x-w/2>=-1e-6 and y-h/2>=-1e-6 and x+w/2<=1+1e-6 and y+h/2<=1+1e-6,(image,row)
   return len(rows)
  labelbytes={name:labels.get(Path(name).stem,b'') for name in import_names}
  counts={name:check_label(name,labelbytes[name]) for name in import_names}
  return labelbytes,counts

CVAT_MISSING={};CVAT_EXISTING={};cvat_counts=[]
for cohort,(added,existing) in ZIP_NAMES.items():
 a,ac=import_cvat_zip(INPUT_DIR/added)
 b,bc=import_cvat_zip(INPUT_DIR/existing)
 assert set(a)==MISSING_NAMES[cohort],('11-image set mismatch',cohort,set(a))
 expected_flagged=set(EXPECTED_SOURCE[EXPECTED_SOURCE.cohort==cohort].image)
 assert set(b)==expected_flagged,('22-image set mismatch',cohort,sorted(set(b)^expected_flagged))
 assert not (set(a)&set(b))
 assert len(a)==EXPECTED_ADDED[cohort][0] and sum(ac.values())==EXPECTED_ADDED[cohort][1]
 assert len(b)==(20 if cohort=='internal' else 2)
 assert sum(bc.values())==(34 if cohort=='internal' else 6)
 CVAT_MISSING[cohort]=a;CVAT_EXISTING[cohort]=b
 for name,n in bc.items():cvat_counts.append(dict(cohort=cohort,image=name,original_flagged=int(EXPECTED_SOURCE[(EXPECTED_SOURCE.cohort==cohort)&(EXPECTED_SOURCE.image==name)].original_boxes.iloc[0]),final_boxes=n))
flagged_count_df=pd.DataFrame(cvat_counts)
flagged_count_df['box_delta']=flagged_count_df.final_boxes-flagged_count_df.original_flagged
flagged_count_df.to_csv(OUT/'Step08C_22_Flagged_Image_Box_Changes.csv',index=False)
assert int(flagged_count_df[flagged_count_df.cohort=='internal'].final_boxes.sum())==34
assert sum(flagged_count_df[flagged_count_df.cohort=='internal'].final_boxes==0)==7
print('Imported 33 CVAT images; 22 new lesion boxes added to prior 11, final existing-box review:',sum(flagged_count_df.final_boxes),'boxes in 22 radiographs.')
print(flagged_count_df[['cohort','original_flagged','final_boxes','box_delta']].groupby('cohort').sum())

STAGE=Path('/content/step08d_staging')
if STAGE.exists():shutil.rmtree(STAGE)
records=[];expected_images={};YAMLS={}
for cohort,src in ORIGINAL.items():
 ims=sorted(x for x in (src/'images/test').iterdir() if x.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'})
 assert len(ims)==EXPECTED_COUNTS[cohort][0]
 expected_images[cohort]={x.name for x in ims}
 totals=Counter()
 for p in ims:
  lab=src/'labels/test'/f'{p.stem}.txt'
  assert lab.exists(),('Missing original YOLO file',p.name)
  ref=ORIGINAL_MANIFEST[(ORIGINAL_MANIFEST.cohort==cohort)&(ORIGINAL_MANIFEST.image==p.name)]
  assert len(ref)==1,('Image not in locked manifest',cohort,p.name)
  rr=ref.iloc[0]
  assert sha256(p)==rr.image_sha256 and sha256(lab)==rr.label_sha256,('STOP: original dataset has changed',p.name)
  original=lab.read_bytes();n_original=len([v for v in original.decode().splitlines() if v.strip()])
  if p.name in CVAT_EXISTING[cohort]:
   expected_original=int(EXPECTED_SOURCE[(EXPECTED_SOURCE.cohort==cohort)&(EXPECTED_SOURCE.image==p.name)].original_boxes.iloc[0])
   assert n_original==expected_original,('Original source-box count does not match expert tracker',p.name)
  if p.name in CVAT_MISSING[cohort]:assert n_original==0,(cohort,p.name)
  missing=CVAT_MISSING[cohort].get(p.name,original)
  combined=CVAT_EXISTING[cohort].get(p.name,missing)
  assert not (p.name in CVAT_MISSING[cohort] and p.name in CVAT_EXISTING[cohort])
  d={'source':original,'missing_only':missing,'combined_adjudicated':combined}
  nr={k:len([x for x in v.decode().splitlines() if x.strip()]) for k,v in d.items()}
  for variant,blob in d.items():
   folder=STAGE/cohort/variant
   (folder/'images/test').mkdir(parents=True,exist_ok=True)
   (folder/'labels/test').mkdir(parents=True,exist_ok=True)
   dest=folder/'images/test'/p.name
   # One safe copy per variant, no edits to source files.
   shutil.copyfile(p,dest)
   (folder/'labels/test'/f'{p.stem}.txt').write_bytes(blob)  # zero-byte TXT is valid for negative images
   assert sha256(dest)==rr.image_sha256
   assert len([v for v in blob.decode().splitlines() if v.strip()])==nr[variant]
   totals[variant]+=nr[variant]
  records.append(dict(cohort=cohort,image=p.name,source_boxes=nr['source'],missing_only_boxes=nr['missing_only'],combined_adjudicated_boxes=nr['combined_adjudicated'],source_image_sha256=rr.image_sha256,source_label_sha256=rr.label_sha256,final_label_sha256=sha256(STAGE/cohort/'combined_adjudicated'/'labels/test'/f'{p.stem}.txt'),changed_by_11=int(p.name in CVAT_MISSING[cohort]),changed_by_22=int(p.name in CVAT_EXISTING[cohort])))
 assert totals['source']==EXPECTED_COUNTS[cohort][1]
 assert totals['missing_only']==(115 if cohort=='internal' else 70)
 assert totals['combined_adjudicated']==(47 if cohort=='internal' else 74)
 for variant in ['source','missing_only','combined_adjudicated']:
  folder=STAGE/cohort/variant
  spec={'path':str(folder),'train':'images/test','val':'images/test','test':'images/test','names':{0:'caries'},'nc':1}
  ypath=OUT/f'{cohort}_{variant}.yaml';ypath.write_text(yaml.safe_dump(spec,sort_keys=False))
  YAMLS[(cohort,variant)]=ypath
 print('PASS',cohort,'n_images',len(ims),'boxes',dict(totals))
record_df=pd.DataFrame(records)
assert len(record_df)==96 and sum(record_df.changed_by_11)==11 and sum(record_df.changed_by_22)==22
record_df.to_csv(OUT/'Final_96_Image_Correction_Manifest.csv',index=False)
print('PASS: all source images and original GT hashes unchanged. 33 corrected images are disjoint.')


## 3. Revalidate three reference variants using the same frozen weights and model-specific native validator (conf .001 and .25)
The model weights, images, NMS settings, and test split are identical. Only the reference YOLO annotations change. Source baseline must match Step 01B.

In [ ]:
IMGSZ=1024;NMS_IOU=0.70;MAX_DET=300;IOU_LEVELS=np.round(np.arange(.50,.951,.05),2)
def as_numpy(x):
 return x.detach().cpu().numpy().copy() if isinstance(x,torch.Tensor) else np.asarray(x).copy()
def capture_run(cohort,variant,conf):
 stats=[];state={'previous':0}
 def callback(v):
  s=v.metrics.stats;names=list(v.metrics.box.image_metrics.keys());n=len(s['tp'])
  assert len(names)==n
  for i in range(state['previous'],n):
   tp=as_numpy(s['tp'][i]).astype(bool).reshape(-1,10)
   scores=as_numpy(s['conf'][i]).astype(float).reshape(-1)
   target=as_numpy(s['target_cls'][i]).astype(int).reshape(-1)
   pc=as_numpy(s['pred_cls'][i]).astype(int).reshape(-1)
   assert len(scores)==len(tp)==len(pc) and np.all(pc==0) and np.all(target==0)
   stats.append(dict(image=names[i],tp=tp,scores=scores,pred_cls=pc,n_gt=len(target)))
  state['previous']=n
 model=YOLO(str(WEIGHTS));model.add_callback('on_val_batch_end',callback)
 yamlpath=YAMLS[(cohort,variant)]
 metrics=model.val(data=str(yamlpath),split='test',imgsz=IMGSZ,conf=conf,iou=NMS_IOU,max_det=MAX_DET,
                   device=0,half=False,workers=0,batch=4,save=False,save_txt=False,save_json=False,
                   plots=False,project=str(OUT/'validation_logs'),name=f'{cohort}_{variant}_{conf}',exist_ok=False,verbose=False)
 assert len(stats)==EXPECTED_COUNTS[cohort][0]
 assert set(x['image'] for x in stats)==set(record_df[record_df.cohort==cohort].image)
 expected=dict(zip(record_df[record_df.cohort==cohort].image,record_df[record_df.cohort==cohort][{'source':'source_boxes','missing_only':'missing_only_boxes','combined_adjudicated':'combined_adjudicated_boxes'}[variant]]))
 assert all(x['n_gt']==expected[x['image']] for x in stats),'STOP: labels seen by Ultralytics are not those audited.'
 return stats,dict(cohort=cohort,variant=variant,conf=conf,mAP50=float(metrics.box.map50),mAP50_95=float(metrics.box.map),model_precision=float(metrics.box.mp),model_recall=float(metrics.box.mr),n_predictions=sum(len(x['scores']) for x in stats))
def native_ap(items):
 tp=np.concatenate([a['tp'] for a in items]);scores=np.concatenate([a['scores'] for a in items]);pc=np.concatenate([a['pred_cls'] for a in items]);gt=np.zeros(sum(a['n_gt'] for a in items),dtype=int)
 if len(gt)==0:return np.full(10,np.nan)
 if len(scores)==0:return np.zeros(10)
 v=ap_per_class(tp,scores,pc,gt,plot=False)[5]
 assert v.shape==(1,10)
 return v[0]
STAT={};results=[]
for cohort in ['internal','external']:
 for variant in ['source','missing_only','combined_adjudicated']:
  for conf in [0.001,0.25]:
   print('VALIDATING',cohort,variant,conf)
   st,r=capture_run(cohort,variant,conf);STAT[(cohort,variant,conf)]=st;results.append(r)
res=pd.DataFrame(results);res.to_csv(OUT/'official_validator_metrics.csv',index=False)
print(res)
# Verify the source-defined validator metrics against Step 01B (exact checkpoint).
ir=res[(res.cohort=='internal')&(res.variant=='source')&(res.conf==0.001)].iloc[0]
er=res[(res.cohort=='external')&(res.variant=='source')&(res.conf==0.001)].iloc[0]
assert abs(ir.mAP50-0.497526)<.0001 and abs(ir.mAP50_95-0.245775)<.0001,'STOP: original internal baseline is not reproduced.'
assert abs(er.mAP50)<1e-8 and abs(er.mAP50_95)<1e-8,'STOP: original external baseline mismatch.'
print('PASS: original AP reproduces Step 01B snapshot.')


## 4. Reconstruct AP, check invariance of predictions, and compute fixed-threshold TP/FP/FN
All reconstructed AP must match official Ultralytics AP; source operating point must match the audited Step 01B numbers.

In [ ]:
ap_parity=[]; per_image=[]; summaries=[];ap_by_iou=[]
def ratio(a,b):return float(a/b) if b else float('nan')
for cohort in ['internal','external']:
 for variant in ['source','missing_only','combined_adjudicated']:
  x=res[(res.cohort==cohort)&(res.variant==variant)&(res.conf==0.001)].iloc[0]
  aps=native_ap(STAT[(cohort,variant,0.001)])
  d1=abs(aps[0]-x.mAP50);d2=abs(np.mean(aps)-x.mAP50_95)
  ap_parity.append(dict(cohort=cohort,variant=variant,AP50_validator=x.mAP50,AP50_reconstructed=aps[0],AP5095_validator=x.mAP50_95,AP5095_reconstructed=np.mean(aps),difference50=d1,difference5095=d2,pass_parity=d1<1e-5 and d2<1e-5))
  for i,a in enumerate(aps):ap_by_iou.append(dict(cohort=cohort,variant=variant,iou=IOU_LEVELS[i],AP=float(a)))
  items=STAT[(cohort,variant,0.25)]
  for t in items:
   tp=int(t['tp'][:,0].sum());nt=t['n_gt'];npr=len(t['scores'])
   per_image.append(dict(cohort=cohort,variant=variant,image=t['image'],nGT=nt,nPred=npr,TP=tp,FP=npr-tp,FN=nt-tp,image_GT_positive=int(nt>0),image_pred_positive=int(npr>0)))
  d=pd.DataFrame([v for v in per_image if v['cohort']==cohort and v['variant']==variant]);tp=int(d.TP.sum());fp=int(d.FP.sum());fn=int(d.FN.sum())
  a=int(((d.image_GT_positive==1)&(d.image_pred_positive==1)).sum());b=int(((d.image_GT_positive==0)&(d.image_pred_positive==1)).sum());c=int(((d.image_GT_positive==0)&(d.image_pred_positive==0)).sum());e=int(((d.image_GT_positive==1)&(d.image_pred_positive==0)).sum())
  summaries.append(dict(cohort=cohort,variant=variant,nImages=len(d),nGT=int(d.nGT.sum()),nPred=int(d.nPred.sum()),lesion_TP=tp,lesion_FP=fp,lesion_FN=fn,lesion_recall=ratio(tp,tp+fn),lesion_precision=ratio(tp,tp+fp),lesion_F1=ratio(2*tp,2*tp+fp+fn),image_TP=a,image_FP=b,image_TN=c,image_FN=e,image_sensitivity=ratio(a,a+e),image_specificity=ratio(c,c+b),image_accuracy=ratio(a+c,len(d)),image_F1=ratio(2*a,2*a+b+e)))
par=pd.DataFrame(ap_parity);assert par.pass_parity.all(),'STOP: AP reconstruction does not match official validator.'
par.to_csv(OUT/'mandatory_AP_parity.csv',index=False)
pd.DataFrame(ap_by_iou).to_csv(OUT/'AP_by_IoU_original_vs_corrected.csv',index=False)
per=pd.DataFrame(per_image);per.to_csv(OUT/'fixed_threshold_per_image.csv',index=False)
summ=pd.DataFrame(summaries);summ.to_csv(OUT/'fixed_threshold_and_image_level_summary.csv',index=False)
# Predictions should be identical even with GT replaced (same model and images).
for cohort in ['internal','external']:
 for conf in [.001,.25]:
  oa={x['image']:x['scores'] for x in STAT[(cohort,'source',conf)]}
  for v in ['missing_only','combined_adjudicated']:
   ca={x['image']:x['scores'] for x in STAT[(cohort,v,conf)]}
   assert all((len(oa[n])==len(ca[n]) and np.allclose(oa[n],ca[n],rtol=1e-6,atol=2e-6)) for n in oa),('STOP: predictions changed across label variants',cohort,v,conf)
# Historical Step 01B fixed-threshold totals provide an additional independent gate.
for cohort, expected_tp,expected_fp,expected_fn in [('internal',9,6,99),('external',0,6,55)]:
 orig=summ[(summ.cohort==cohort)&(summ.variant=='source')].iloc[0]
 assert (orig.lesion_TP,orig.lesion_FP,orig.lesion_FN)==(expected_tp,expected_fp,expected_fn), ('STOP: original operating point is not reproduced',cohort,orig.to_dict())
print('PASS: 6/6 AP parity checks, Step 01B source TP/FP/FN, and unchanged prediction scores across three reference variants.');display(summ)


## 5. Paired bootstrap uncertainty (2,000 resamples per cohort)
Resampling unit is the radiograph, paired across all three reference variants.

In [ ]:
# Paired bootstrap on the SAME 96 image identities (per cohort); compare all three GT variants.
RNGSEED=42;B=2000;boot_rows=[]
for cohort in ['internal','external']:
 names=sorted(record_df[record_df.cohort==cohort].image)
 base={v:{x['image']:x for x in STAT[(cohort,v,0.001)]} for v in ['source','missing_only','combined_adjudicated']}
 rng=np.random.default_rng(RNGSEED)
 for rep in range(B):
  chosen=[names[k] for k in rng.integers(0,len(names),len(names))]
  ap={v:native_ap([base[v][n] for n in chosen]) for v in base}
  row={'cohort':cohort,'replicate':rep+1}
  for v,aps in ap.items():
   row[f'AP50_{v}']=float(aps[0]);row[f'AP5095_{v}']=float(np.mean(aps))
  for a,b in [('missing_only','source'),('combined_adjudicated','source'),('combined_adjudicated','missing_only')]:
   row[f'AP50_diff_{a}_minus_{b}']=float(ap[a][0]-ap[b][0])
   row[f'AP5095_diff_{a}_minus_{b}']=float(np.mean(ap[a])-np.mean(ap[b]))
  boot_rows.append(row)
 print('Paired bootstrap completed:',cohort)
boot=pd.DataFrame(boot_rows)
boot.to_csv(OUT/'paired_bootstrap_2000.csv',index=False)
ci=[]
for cohort,d in boot.groupby('cohort'):
 for field in [x for x in d.columns if x not in ('cohort','replicate')]:
  x=d[field].dropna().to_numpy(float)
  ci.append(dict(cohort=cohort,metric=field,CI_2_5=float(np.quantile(x,.025)) if len(x) else np.nan,CI_97_5=float(np.quantile(x,.975)) if len(x) else np.nan,n_valid=len(x)))
pd.DataFrame(ci).to_csv(OUT/'paired_bootstrap_95CI.csv',index=False)
print('Complete 95% paired bootstrap uncertainty for original, 11-only and all 33 images.')


## 6. Side-by-side source and corrected annotation overlay audit for 33 images
Overlays are for private expert QC and manuscript figure selection subject to image rights; no model predictions are used to correct GT.

In [ ]:
# Expert-only overlays: source boxes in blue, approved GT in red, separate image panels.
# Keep these overlays private unless image-source license permits publication.
OVER=OUT/'review_overlays';OVER.mkdir(exist_ok=True)
for rec in record_df.itertuples(index=False):
 if rec.changed_by_11==0 and rec.changed_by_22==0:continue
 imgpath=ORIGINAL[rec.cohort]/'images/test'/rec.image
 with Image.open(imgpath) as im:
  im=im.convert('RGB');W,H=im.size
  panels=[]
  for label,variant,color in [('SOURCE','source',(15,112,230)),('EXPERT-APPROVED','combined_adjudicated',(210,30,65))]:
   canvas=im.copy();draw=ImageDraw.Draw(canvas)
   p=STAGE/rec.cohort/variant/'labels/test'/f'{Path(rec.image).stem}.txt'
   for k,line in enumerate(p.read_text().splitlines(),1):
    if not line.strip():continue
    _,xc,yc,w,h=map(float,line.split())
    xy=((xc-w/2)*W,(yc-h/2)*H,(xc+w/2)*W,(yc+h/2)*H)
    draw.rectangle(xy,outline=color,width=max(2,round(min(W,H)/320)))
    draw.text((xy[0],max(0,xy[1]-17)),str(k),fill=color)
   draw.text((18,18),f'{label} | n={sum(bool(x.strip()) for x in p.read_text().splitlines())}',fill=color)
   panels.append(canvas)
  composed=Image.new('RGB',(2*W,H),(255,255,255))
  composed.paste(panels[0],(0,0));composed.paste(panels[1],(W,0))
  composed.thumbnail((3400,1600))
  composed.save(OVER/f'{rec.cohort}__{Path(rec.image).stem}.jpg',quality=85)
print('Review overlays:',len(list(OVER.glob('*.jpg'))),'(derived previews; do not publish without rights review).')


## 7. Export complete evidence ZIP and run README
Upload ZIP back into the chat for statistical analysis and reviewer-response finalization.

In [ ]:
summary={
 'analysis':'Exploratory paired sensitivity: source-defined reference versus 11 previously source-negative images with 22 approved added boxes versus combined adjudication of 33 images (11 missing-box and 22 legacy-box images).',
 'model_sha256':EXPECTED_MODEL,'source_images_modified':False,'source_annotation_files_modified':False,
 'ultralytics_version':ultralytics.__version__,'imgsz':IMGSZ,'confidence_AP':0.001,'confidence_operating':0.25,
 'NMS_iou':NMS_IOU,'max_detections':MAX_DET,'bootstrap_replicates':B,
 'caveat':'Expert-approved correction applies only to 33 flagged images, not all 96 radiographs or training annotation sets. Reviewer asserted final approval in conversation; retain original written specialist approval.',
 'reproducibility':'Checkpoints are held in the project Drive and not bundled for redistribution; data source licenses may limit release.'}
(OUT/'README_ANALYSIS_LIMITATIONS.json').write_text(json.dumps(summary,indent=2))
# Plain-language QC report, without asserting corrected GT as the clinical gold standard.
s=pd.DataFrame(records).groupby('cohort')[['source_boxes','missing_only_boxes','combined_adjudicated_boxes']].sum()
with open(OUT/'CORRECTED_GT_README.md','w') as fh:
 fh.write('# Expert-corrected GT sensitivity analysis (frozen model)\n\n')
 fh.write('Source-defined original GT is unchanged. Two successive correction interventions: 11 previously unlabeled images, 22 subsequently adjudicated legacy-box images.\n\n')
 fh.write('```\n'+s.to_string()+'\n```\n\n')
 fh.write('This is an exploratory sensitivity analysis, not evidence of corrected labels on all available images.\n')
archive=OUT/'Step_08D_Final_Adjudicated_GT_Results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for p in OUT.rglob('*'):
  if not p.is_file() or p==archive or 'validation_logs' in p.parts:continue
  z.write(p,p.relative_to(OUT))
print('COMPLETE',archive,'bytes',archive.stat().st_size)
print(res[['cohort','variant','conf','mAP50','mAP50_95']].to_string(index=False))
